# AutoGen Framework vs Direct API Coding

## Context
This guide maps what you learned in the `llm_engineering` course (direct OpenAI SDK usage) to how AutoGen works — so you can apply your foundation to AutoGen projects.

---

## What You Learned Here → How It Maps to AutoGen

### 1. Single LLM Call
**Direct API (course style):**
```python
response = azure_openai.chat.completions.create(
    model="gpt-4o",
    messages=[{"role": "user", "content": "Hello"}]
)
```

**AutoGen equivalent:**
```python
from autogen_agentchat.agents import AssistantAgent

agent = AssistantAgent(name="assistant", model_client=model_client)
await agent.run(task="Hello")
```
AutoGen wraps the same API call — it's doing the same thing under the hood.

---

### 2. Conversation History
**Direct API — you managed manually (week2/day3.ipynb):**
```python
def chat(message, history):
    messages = [{"role": "system", "content": system_message}]
    for h in history:
        messages.append({"role": "user", "content": h[0]})
        messages.append({"role": "assistant", "content": h[1]})
```

**AutoGen equivalent:**
AutoGen agents **remember history automatically**. No manual list management needed.

---

### 3. Tool Calling
**Direct API — you manually checked finish_reason (week2/day4.ipynb):**
```python
if response.choices[0].finish_reason == "tool_calls":
    tool_responses = handle_tool_calls(response.choices[0].message)
    messages.append(tool_responses)
```

**AutoGen equivalent:**
```python
agent = AssistantAgent(
    name="assistant",
    model_client=model_client,
    tools=[get_ticket_price, set_ticket_price]  # just pass functions directly!
)
```
AutoGen handles `tool_calls` detection, execution, and response **automatically**.
Your entire `handle_tool_calls()` function is replaced by the framework.

---

### 4. Multi-Agent (new concept for AutoGen projects)
**Direct API:** You had ONE AI responding to the user.

**AutoGen adds:** Multiple AI agents talking to each other:
```python
from autogen_agentchat.teams import RoundRobinGroupChat

researcher = AssistantAgent("researcher", ...)
writer     = AssistantAgent("writer", ...)
critic     = AssistantAgent("critic", ...)

team = RoundRobinGroupChat([researcher, writer, critic])
await team.run(task="Write a report on climate change")
```
researcher gathers info → writer drafts → critic reviews → automatically, in a loop.

---

### 5. Streaming
**Direct API (week2/day3.ipynb):**
```python
stream = openai.chat.completions.create(stream=True, ...)
for chunk in stream:
    if chunk.choices:   # guard needed for Azure!
        yield chunk.choices[0].delta.content
```

**AutoGen equivalent:**
```python
async for message in agent.run_stream(task="..."):
    print(message)   # streaming handled by framework
```

---

## The Big Picture Mental Model

| This Course (Direct API)   | AutoGen Equivalent           |
|----------------------------|------------------------------|
| Messages list              | Agent memory (automatic)     |
| System prompt              | `agent.system_message`       |
| Tool calling loop          | `tools=[]` parameter         |
| Manual history management  | Built-in context window      |
| 1 model responding         | Team of agents               |
| Gradio UI                  | Can still use Gradio on top! |

> Your knowledge from this course covers ~70% of what AutoGen does internally.

---

## What's New in AutoGen (the 30% you still need to learn)

1. **Agent types**
   - `AssistantAgent` — the AI
   - `UserProxyAgent` — represents the human / runs code
   - `CodeExecutorAgent` — executes generated code automatically

2. **Team patterns**
   - `RoundRobinGroupChat` — agents take turns in order
   - `SelectorGroupChat` — a manager picks who speaks next
   - `Swarm` — agents hand off tasks to each other

3. **Termination conditions** — when should the agent loop stop?
   ```python
   from autogen_agentchat.conditions import TextMentionTermination
   termination = TextMentionTermination("DONE")
   ```

4. **Human-in-the-loop** — where should a human approve/intervene?

5. **`async/await`** — AutoGen is fully async (Python async programming)
   ```python
   import asyncio
   asyncio.run(main())
   ```

---

## Key Insight

Most AutoGen users don't understand what's happening behind the scenes.
Because you built everything from scratch in this course, you do — which makes you a much stronger AutoGen developer.

---

## References
- [AutoGen Docs](https://microsoft.github.io/autogen/)
- Course notebooks: `week2/day3.ipynb` (history/streaming), `week2/day4.ipynb` (tool calling)
